# LlamaIndex Practice

Welcome to your hands-on practice with LlamaIndex! In this notebook, you'll learn how to:
- Create custom tools for AI agents
- Build a functional agent using Azure OpenAI
- Integrate Model Context Protocol (MCP) tools
- Implement observability and debugging
- Complete a creative challenge to apply your knowledge

## What You'll Build

In this notebook, you'll create an intelligent **Travel Planning Assistant** that can:
- Search for flight information
- Get weather forecasts for destinations
- Convert currencies for travel budgeting
- Use Model Context Protocol (MCP) for enhanced capabilities

By the end, you'll understand the complete agent creation flow with LlamaIndex and be ready to build your own creative agents!

## Step 1: Initial Setup & Installation

First, let's install the required packages. We need:
- **llama-index-core**: Core LlamaIndex framework (agents, tools, workflows)
- **llama-index-llms-openai**: OpenAI-compatible LLM integration (works with Azure OpenAI's v1 endpoint)
- **python-dotenv**: Loads credentials from a `.env` file

In [1]:
%pip install -q llama-index-core llama-index-llms-openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


## Step 2: Configure Azure OpenAI Credentials

You'll need an Azure OpenAI API key, endpoint, and model deployment. Store them in a `.env` file (in this folder or any parent folder); if a value is missing, the cell below prompts for it.
```
AZURE_OPENAI_API_KEY=<your-key>
AZURE_OPENAI_ENDPOINT=https://<resource>.openai.azure.com/openai/v1/
AZURE_OPENAI_DEPLOYMENT=gpt-5.4-mini
```

In [2]:
import getpass
import os

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

if not os.environ.get("AZURE_OPENAI_API_KEY"):
    os.environ["AZURE_OPENAI_API_KEY"] = getpass.getpass("Enter your Azure OpenAI API key: ")
if not os.environ.get("AZURE_OPENAI_ENDPOINT"):
    os.environ["AZURE_OPENAI_ENDPOINT"] = input("Enter your Azure OpenAI endpoint (https://<resource>.openai.azure.com/): ")

# Accept either the resource URL or the full v1 base URL
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/").removesuffix("/openai/v1") + "/openai/v1/"
AZURE_OPENAI_DEPLOYMENT = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-5.4-mini")
print(f"Using Azure OpenAI deployment: {AZURE_OPENAI_DEPLOYMENT}")

Using Azure OpenAI deployment: gpt-5.4-mini


## Step 3: Create Custom Tools

Tools are functions that your agent can call to perform specific tasks. Let's create travel-related tools.

**Key Concepts:**
- Each tool is a Python function with clear docstrings
- Type hints help the LLM understand parameter types
- Docstrings describe what the tool does (the LLM reads this!)

In [3]:
from datetime import datetime, timedelta
import random


def search_flights(origin: str, destination: str, date: str) -> str:
    """Search for available flights between two cities on a specific date.

    Args:
        origin: The departure city (e.g., 'New York')
        destination: The arrival city (e.g., 'Paris')
        date: The travel date in YYYY-MM-DD format

    Returns:
        A string with flight information including price and duration
    """
    # Simulate flight search
    flight_number = f"FL{random.randint(100, 999)}"
    price = random.randint(300, 1500)
    duration = random.randint(4, 15)

    return f"Found flight {flight_number} from {origin} to {destination} on {date}. Price: ${price}, Duration: {duration}h"


def get_weather(city: str, days_ahead: int = 0) -> str:
    """Get weather forecast for a city.

    Args:
        city: The city name to get weather for
        days_ahead: Number of days ahead to forecast (0 = today, 1 = tomorrow, etc.)

    Returns:
        A string describing the weather conditions
    """
    conditions = ["Sunny", "Partly Cloudy", "Cloudy", "Rainy", "Stormy"]
    condition = random.choice(conditions)
    temp = random.randint(15, 30)
    forecast_date = datetime.now() + timedelta(days=days_ahead)

    return f"Weather in {city} on {forecast_date.strftime('%Y-%m-%d')}: {condition}, {temp}°C"


def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert an amount from one currency to another.

    Args:
        amount: The amount to convert
        from_currency: Source currency code (e.g., 'USD')
        to_currency: Target currency code (e.g., 'EUR')

    Returns:
        A string with the converted amount
    """
    # Simplified exchange rates (in practice, you'd use a real API)
    rates = {
        ("USD", "EUR"): 0.92,
        ("USD", "GBP"): 0.79,
        ("USD", "JPY"): 149.50,
        ("EUR", "USD"): 1.09,
        ("GBP", "USD"): 1.27,
    }

    rate = rates.get((from_currency, to_currency), 1.0)
    converted = amount * rate

    return f"{amount} {from_currency} = {converted:.2f} {to_currency}"

## Step 4: Initialize the Language Model

Now let's set up Azure OpenAI as our LLM. Azure OpenAI's v1 API is OpenAI-compatible, so we use LlamaIndex's `OpenAI` class with `api_base` pointing at the Azure endpoint.

**Note:** `model` is your deployment name. It must match an OpenAI model name (e.g. `gpt-5.4-mini`, `gpt-5.6-luna`) so LlamaIndex knows the model's context window and tool-calling support.

In [4]:
from llama_index.llms.openai import OpenAI

llm = OpenAI(
    model=AZURE_OPENAI_DEPLOYMENT,
    api_base=AZURE_OPENAI_ENDPOINT,
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
)

## Step 5: Create Your First Agent

A **FunctionAgent** is LlamaIndex's agentic workflow that can:
1. Understand user queries
2. Decide which tools to use
3. Call tools with appropriate parameters
4. Synthesize responses from tool outputs

Let's create our Travel Planning Agent!

In [5]:
from llama_index.core.agent.workflow import FunctionAgent

agent = FunctionAgent(
    tools=[search_flights, get_weather, convert_currency],
    llm=llm,
    verbose=True  # This helps us see what the agent is doing
)

## Step 6: Create an Observability Helper

Before testing our agent, let's create a helper function that gives us **detailed visibility** into what the agent is doing.

**Why Observability Matters:**
- 🔍 See which tools the agent chooses
- 📊 Inspect the inputs passed to each tool
- ✅ Verify the outputs returned
- 🐛 Debug issues when the agent doesn't behave as expected

The `run_agent_verbose()` function will:
1. Display the user's query
2. Stream events as the agent works
3. Print details about each tool call
4. Return the final response

This is crucial for understanding and debugging agent behavior!

In [6]:
from llama_index.core.agent.workflow import ToolCallResult


async def run_agent_verbose(query: str):
    """Run the agent and print detailed information about tool calls."""
    print(f"🤖 User Query: {query}\n")
    print("=" * 60)

    handler = agent.run(query)
    async for event in handler.stream_events():
        if isinstance(event, ToolCallResult):
            print(f"🔧 Tool Used: {event.tool_name}")
            print(f"📥 Input: {event.tool_kwargs}")
            print(f"📤 Output: {event.tool_output}")
            print("-" * 60)

    result = await handler
    print(f"\n✨ Final Response:\n{result}\n")
    return result

## Step 7: Test Your Agent with Simple Queries

Let's start with basic queries to see how the agent uses tools:

In [7]:
response = await run_agent_verbose(
    "What's the weather like in Tokyo today?"
)

🤖 User Query: What's the weather like in Tokyo today?

[tick] add: AgentWorkflowStartEvent(user_msg="What's the weather like in Tokyo today?", chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
[init_run:0] started from AgentWorkflowStartEvent


[init_run:0] complete with AgentInput
[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="What's the weather like in Tokyo today?")])], current_agent_name='Age...
[setup_agent:0] started from AgentInput
[setup_agent:0] complete with AgentSetup
[tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="What's the weather like in Tokyo today?")])], current_agent_name='Age...
[run_agent_step:0] started from AgentSetup


[run_agent_step:0] complete with AgentOutput
[tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={'tool_calls': [ChoiceDeltaToolCall(index=0, id='call_cacOqHvqh8TYpNRZgzjiIaOV', function=ChoiceDeltaTo...
[parse_agent_output:0] started from AgentOutput
[tick] add: ToolCall(tool_name='get_weather', tool_kwargs={'city': 'Tokyo', 'days_ahead': 0}, tool_id='call_cacOqHvqh8TYpNRZgzjiIaOV')
[call_tool:0] started from ToolCall
[parse_agent_output:0] complete with no result
🔧 Tool Used: get_weather
📥 Input: {'city': 'Tokyo', 'days_ahead': 0}
📤 Output: Weather in Tokyo on 2026-11-23: Cloudy, 27°C
------------------------------------------------------------
[call_tool:0] complete with ToolCallResult
[tick] add: ToolCallResult(tool_name='get_weather', tool_kwargs={'city': 'Tokyo', 'days_ahead': 0}, tool_id='call_cacOqHvqh8TYpNRZgzjiIaOV', tool_output=ToolOutput(blocks=[TextBlock(block_type='text', text='We...
[aggregate_tool_results:0] started 

[run_agent_step:0] complete with AgentOutput
[tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Weather in Tokyo today: Cloudy, 27°C.')]), structured_re...
[parse_agent_output:0] started from AgentOutput
[result] StopEvent(result=AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Weather in Tokyo today: Cloudy, 27°C.')...
[parse_agent_output:0] complete with StopEvent

✨ Final Response:
Weather in Tokyo today: Cloudy, 27°C.



In [8]:
# Inspect which tools were called
print("Tool Calls Made:")
for tool_call in response.tool_calls:
    print(f"  - {tool_call}")

Tool Calls Made:
  - tool_name='get_weather' tool_kwargs={'city': 'Tokyo', 'days_ahead': 0} tool_id='call_cacOqHvqh8TYpNRZgzjiIaOV' tool_output=ToolOutput(blocks=[TextBlock(block_type='text', text='Weather in Tokyo on 2026-11-23: Cloudy, 27°C')], tool_name='get_weather', raw_input={'args': (), 'kwargs': {'city': 'Tokyo', 'days_ahead': 0}}, raw_output='Weather in Tokyo on 2026-11-23: Cloudy, 27°C', is_error=False) return_direct=False


## Step 8: Complex Multi-Tool Query

Watch how the agent chains multiple tools together to answer complex questions:

In [9]:
response = await run_agent_verbose(
    "I want to fly from London to Barcelona on 2026-12-15. What's the weather there, and how much is 500 USD in EUR?"
)

🤖 User Query: I want to fly from London to Barcelona on 2026-12-15. What's the weather there, and how much is 500 USD in EUR?

[tick] add: AgentWorkflowStartEvent(user_msg='I want to fly from London to Barcelona on 2026-12-...', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
[init_run:0] started from AgentWorkflowStartEvent
[init_run:0] complete with AgentInput
[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="I want to fly from London to Barcelona on 2026-12-15. What's the weat...
[setup_agent:0] started from AgentInput
[setup_agent:0] complete with AgentSetup
[tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="I want to fly from London to Barcelona on 2026-12-15. What's the weat...
[run_agent_step:0] started from AgentSetup


[run_agent_step:0] complete with AgentOutput
[tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={'tool_calls': [ChoiceDeltaToolCall(index=0, id='call_1lm4grPhs5Q06rErQEYZEVLY', function=ChoiceDeltaTo...
[parse_agent_output:0] started from AgentOutput
[tick] add: ToolCall(tool_name='search_flights', tool_kwargs={'origin': 'London', 'destination': 'Barcelona', 'date': '2026-12-15'}, tool_id='call_1lm4grPhs5Q06rErQEYZEVLY')
[call_tool:0] started from ToolCall
[parse_agent_output:0] complete with no result
[tick] add: ToolCall(tool_name='get_weather', tool_kwargs={'city': 'Barcelona', 'days_ahead': 0}, tool_id='call_kcGVU0VrIsMST9L6rITNWpSm')
[call_tool:1] started from ToolCall
[tick] add: ToolCall(tool_name='convert_currency', tool_kwargs={'amount': 500, 'from_currency': 'USD', 'to_currency': 'EUR'}, tool_id='call_1tXodTcaTA3sKyteYlKRt3IO')
[call_tool:2] started from ToolCall
🔧 Tool Used: convert_currency
📥 Input: {'amount': 500, 'fro

[run_agent_step:0] complete with AgentOutput
[tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Here’s a quick summary for your trip:\n\n- Flight: Found...
[parse_agent_output:0] started from AgentOutput
[result] StopEvent(result=AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Here’s a quick summary for your trip:\n...
[parse_agent_output:0] complete with StopEvent

✨ Final Response:
Here’s a quick summary for your trip:

- Flight: Found flight FL271 from London to Barcelona on 2026-12-15
  - Price: $609
  - Duration: 15h

- Weather in Barcelona: Sunny, 27°C

- Currency conversion: 500 USD = 460.00 EUR

If you want, I can also help compare flights or suggest a better route.



## Step 9: Managing Context & Memory

By default, `.run()` is stateless (doesn't remember previous conversations). Let's add memory using a **Context** object:

In [10]:
from llama_index.core.workflow import Context

# Create a new agent instance for this demo
memory_agent = FunctionAgent(
    tools=[search_flights, get_weather, convert_currency],
    llm=llm
)

# Create a context to maintain conversation history
ctx = Context(memory_agent)

# First message - introduce yourself
response1 = await memory_agent.run(
    "Hi! I'm planning a trip to Rome in December. My name is Alex.",
    ctx=ctx
)
print(f"Response 1: {response1}\n")

# Second message - agent should remember your name
response2 = await memory_agent.run(
    "What's my name, and what's the weather like in my destination?",
    ctx=ctx
)
print(f"Response 2: {response2}")

Response 1: Hi Alex! Rome in December can be lovely—cooler weather, fewer crowds, and great holiday atmosphere.

If you want, I can help you with any of these:
- flight options to Rome
- December weather in Rome
- trip budgeting and currency conversion
- itinerary ideas

Just tell me what you’d like to look into.



C:\Users\artem.lubkivskyi\Documents\GitHub\Accenture\Day 14\.venv\Lib\site-packages\workflows\context\state_store.py:231: UnserializableKeyWarning: Skipping serialization of known unserializable key: memory -- This is expected but will require this item to be set manually after deserialization.
  warnings.warn(


Response 2: I don’t know your name yet, and I don’t know your destination city either.

If you tell me:
1. your name, and  
2. your destination city,

I can help with the weather there.


## Step 10: Advanced - Model Context Protocol (MCP) Tools

MCP is a standardized protocol for connecting AI models to external tools and data sources. Let's integrate an MCP tool!

**What is MCP?**
- Open protocol for AI-to-tool communication
- Enables standardized tool integration
- Supports local and remote tools

### Creating a Custom MCP-Style Tool

For this practice, we simulate an MCP tool that fetches travel tips with a regular `FunctionTool`.

To load tools from a real MCP server, install `llama-index-tools-mcp` and convert the server's tools into LlamaIndex tools:
```python
# from llama_index.tools.mcp import BasicMCPClient, McpToolSpec
# mcp_client = BasicMCPClient("http://127.0.0.1:8000/mcp")  # or BasicMCPClient("python", args=["server.py"]) for stdio
# mcp_tools = await McpToolSpec(client=mcp_client).to_tool_list_async()
```

In [11]:
from llama_index.core.tools import FunctionTool


def get_travel_tips(destination: str) -> str:
    """Get essential travel tips for a destination.

    Args:
        destination: The city or country to get tips for

    Returns:
        A string with helpful travel tips
    """
    tips_database = {
        "paris": "🇫🇷 Tips for Paris: Try croissants at local boulangeries, use the Metro, visit Eiffel Tower early morning",
        "tokyo": "🇯🇵 Tips for Tokyo: Get a Suica card for transport, try ramen at local shops, visit temples in Asakusa",
        "new york": "🇺🇸 Tips for NYC: Use subway, try pizza and bagels, visit Central Park and Times Square",
        "barcelona": "🇪🇸 Tips for Barcelona: Visit Sagrada Familia, try tapas and paella, walk Las Ramblas",
        "rome": "🇮🇹 Tips for Rome: Visit Colosseum early, try authentic carbonara, throw coin in Trevi Fountain",
    }

    city_lower = destination.lower()
    for key in tips_database:
        if key in city_lower:
            return tips_database[key]

    return f"💡 General tips: Research local customs, learn basic phrases, try local cuisine!"


# Convert to LlamaIndex tool
travel_tips_tool = FunctionTool.from_defaults(fn=get_travel_tips)

# Create agent with MCP-style tool
mcp_agent = FunctionAgent(
    tools=[search_flights, get_weather, convert_currency, travel_tips_tool],
    llm=llm
)

print("✅ Agent with MCP tool created!")

✅ Agent with MCP tool created!


### Testing the Enhanced Agent

In [12]:
async def run_mcp_agent(query: str):
    """Run the MCP-enhanced agent."""
    print(f"🤖 Query: {query}\n")
    handler = mcp_agent.run(query)
    async for event in handler.stream_events():
        if isinstance(event, ToolCallResult):
            print(f"🔧 {event.tool_name} → {event.tool_output}")

    result = await handler
    print(f"\n✨ {result}\n")
    return result


response = await run_mcp_agent(
    "I'm planning a trip to Tokyo. Give me travel tips and check the weather for tomorrow."
)

🤖 Query: I'm planning a trip to Tokyo. Give me travel tips and check the weather for tomorrow.



🔧 get_travel_tips → 🇯🇵 Tips for Tokyo: Get a Suica card for transport, try ramen at local shops, visit temples in Asakusa
🔧 get_weather → Weather in Tokyo on 2026-11-24: Partly Cloudy, 15°C



✨ Here’s a quick trip snapshot for Tokyo:

**Travel tips**
- Get a **Suica card** for easy train and subway travel.
- Try **ramen at local shops** for a great food experience.
- Visit the **temples in Asakusa** for a classic cultural stop.

**Tomorrow’s weather in Tokyo**
- **Partly cloudy**
- **15°C**

If you want, I can also help with a packing list or suggest things to do based on the weather.



## Step 11: Enhanced Observability with Custom Callbacks

Let's implement more detailed observability to track agent performance:

In [13]:
import time
from typing import List


class AgentObserver:
    """Track and display agent performance metrics."""

    def __init__(self):
        self.tool_calls = []
        self.start_time = None
        self.end_time = None

    async def observe_agent(self, query: str, agent: FunctionAgent):
        """Run agent and collect observability data."""
        self.tool_calls = []
        self.start_time = time.time()

        print(f"🎯 Query: {query}\n")
        print("📊 Observability Dashboard")
        print("=" * 60)

        handler = agent.run(query)
        async for event in handler.stream_events():
            if isinstance(event, ToolCallResult):
                self.tool_calls.append({
                    'tool': event.tool_name,
                    'input': event.tool_kwargs,
                    'output': event.tool_output
                })
                print(f"⚙️  Tool Call #{len(self.tool_calls)}: {event.tool_name}")

        result = await handler
        self.end_time = time.time()

        # Display metrics
        print("\n" + "=" * 60)
        print("📈 Performance Metrics:")
        print(f"  ⏱️  Total Time: {self.end_time - self.start_time:.2f}s")
        print(f"  🔧 Tools Called: {len(self.tool_calls)}")
        print(f"  📝 Tool Names: {[t['tool'] for t in self.tool_calls]}")
        print(f"\n💬 Response: {result}")

        return result


# Test the observer
observer = AgentObserver()
await observer.observe_agent(
    "Find flights from Paris to Tokyo on 2026-12-20, check weather, and convert 1000 EUR to JPY",
    mcp_agent
)

🎯 Query: Find flights from Paris to Tokyo on 2026-12-20, check weather, and convert 1000 EUR to JPY

📊 Observability Dashboard


⚙️  Tool Call #1: get_weather
⚙️  Tool Call #2: search_flights
⚙️  Tool Call #3: convert_currency



📈 Performance Metrics:
  ⏱️  Total Time: 2.67s
  🔧 Tools Called: 3
  📝 Tool Names: ['get_weather', 'search_flights', 'convert_currency']

💬 Response: Here’s the info you requested:

- Flights: Found flight **FL523** from **Paris to Tokyo** on **2026-12-20**
  - Price: **$717**
  - Duration: **4h**

- Weather in **Tokyo**:
  - **Stormy, 15°C** on **2026-11-23**

- Currency conversion:
  - **1000 EUR = 1000.00 JPY**

If you want, I can also help compare more flights or suggest travel tips for Tokyo.


AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Here’s the info you requested:\n\n- Flights: Found flight **FL523** from **Paris to Tokyo** on **2026-12-20**\n  - Price: **$717**\n  - Duration: **4h**\n\n- Weather in **Tokyo**:\n  - **Stormy, 15°C** on **2026-11-23**\n\n- Currency conversion:\n  - **1000 EUR = 1000.00 JPY**\n\nIf you want, I can also help compare more flights or suggest travel tips for Tokyo.')]), structured_response=None, current_agent_name='Agent', raw={'id': 'chatcmpl-EnPfcyj2U4cIdw16MZselDGIqUfEa', 'choices': [{'delta': {'audio': None, 'content': None, 'function_call': None, 'refusal': None, 'role': None, 'tool_calls': None}, 'index': 0, 'finish_reason': 'stop', 'logprobs': None, 'content_filter_results': {}}], 'created': 1795445456, 'model': 'gpt-5.4-mini-2026-03-17', 'object': 'chat.completion.chunk', 'moderation': None, 'obfuscation': 'ASoDrO3MFhdc', 'service_tier': 'def

## 🎉 Congratulations!

You've learned how to:
- ✅ Create custom tools for AI agents
- ✅ Build functional agents with LlamaIndex
- ✅ Implement observability and debugging
- ✅ Manage conversation context and memory
- ✅ Integrate MCP-style tools
- ✅ Track agent performance metrics

### Share Your Work:
- Complete the challenge and share your restaurant agent
- Experiment with different tools and domains
- Try integrating real APIs for production use

Happy building! 🚀

## 🎓 Challenge: Build Your Own Agent!

Now it's your turn! Apply what you've learned to create a unique agent.

### Your Mission:
Create a **Restaurant Recommendation Agent** that helps users find and learn about restaurants.

### Requirements:
1. **Create at least 3 custom tools:**
   - `find_restaurants(cuisine: str, city: str) -> str`: Find restaurants by cuisine type
   - `get_restaurant_details(restaurant_name: str) -> str`: Get details like hours, price range
   - `make_reservation(restaurant_name: str, date: str, party_size: int) -> str`: Simulate booking

2. **Add a bonus tool** (get creative!):
   - Menu translator
   - Dietary restriction filter
   - Restaurant reviews summarizer
   - Distance calculator
   - Or your own idea!

3. **Implement:**
   - Agent with all your tools
   - Observability (print tool calls)
   - Context/memory for conversation
   - Test with at least 2 complex queries

4. **Bonus Points:**
   - Make it work with real APIs (like Yelp or Google Places)
   - Add error handling
   - Create a user-friendly interface
   - Implement caching for repeated queries

### Tips:
- Use clear docstrings - the LLM reads them!
- Test each tool individually first
- Start simple, then add complexity
- Use type hints for better tool understanding
- Think about real-world use cases

### Example Test Queries:
- "Find Italian restaurants in Boston and get details about the top one"
- "I'm vegetarian. Find a good restaurant in Seattle and make a reservation for 4 people tomorrow"
- "Recommend a restaurant with outdoor seating, check if they're open now, and book a table"

### Your Solution Space

Write your code below:

In [14]:
# Small fake restaurant database
RESTAURANTS = {
    "Trattoria Nonna": {"city": "riga", "cuisine": "italian", "price": "€€", "hours": "12:00-23:00",
                        "diets": ["vegetarian"], "max_party": 8},
    "Forno Verde": {"city": "riga", "cuisine": "italian", "price": "€", "hours": "11:00-21:00",
                    "diets": ["vegetarian", "vegan"], "max_party": 6},
    "Sakura House": {"city": "riga", "cuisine": "japanese", "price": "€€€", "hours": "17:00-23:00",
                     "diets": ["gluten-free"], "max_party": 10},
    "Green Bowl": {"city": "tallinn", "cuisine": "vegan", "price": "€", "hours": "09:00-20:00",
                   "diets": ["vegetarian", "vegan", "gluten-free"], "max_party": 4},
}
bookings = []


def find_restaurants(cuisine: str, city: str) -> str:
    """Find restaurants of a cuisine type in a city.

    Args:
        cuisine: Cuisine type, e.g. 'italian', 'japanese', 'vegan'
        city: City name, e.g. 'Riga'
    """
    found = [name for name, r in RESTAURANTS.items()
             if r["city"] == city.lower() and r["cuisine"] == cuisine.lower()]
    if not found:
        return f"No {cuisine} restaurants found in {city}."
    return f"{cuisine.title()} restaurants in {city.title()}: " + ", ".join(found)


def get_restaurant_details(restaurant_name: str) -> str:
    """Get opening hours, price range and diet options of a restaurant.

    Args:
        restaurant_name: Exact restaurant name
    """
    r = RESTAURANTS.get(restaurant_name)
    if r is None:
        return f"Unknown restaurant '{restaurant_name}'."
    return (f"{restaurant_name}: {r['cuisine']}, price {r['price']}, open {r['hours']}, "
            f"diets: {', '.join(r['diets'])}, tables up to {r['max_party']} people")


def make_reservation(restaurant_name: str, date: str, party_size: int) -> str:
    """Book a table.

    Args:
        restaurant_name: Exact restaurant name
        date: Date in YYYY-MM-DD format
        party_size: Number of people
    """
    r = RESTAURANTS.get(restaurant_name)
    if r is None:
        return f"Cannot book: unknown restaurant '{restaurant_name}'."
    if party_size < 1 or party_size > r["max_party"]:
        return f"Cannot book: {restaurant_name} takes 1 to {r['max_party']} people, asked for {party_size}."
    try:
        datetime.strptime(date, "%Y-%m-%d")
    except ValueError:
        return f"Cannot book: '{date}' is not a YYYY-MM-DD date."
    code = f"R{len(bookings) + 1:03d}"
    bookings.append((code, restaurant_name, date, party_size))
    return f"Booked {restaurant_name} on {date} for {party_size} people. Confirmation code {code}."


# Bonus tool: dietary filter
def filter_by_diet(city: str, diet: str) -> str:
    """List restaurants in a city that support a diet (vegetarian, vegan, gluten-free).

    Args:
        city: City name
        diet: Diet name
    """
    found = [name for name, r in RESTAURANTS.items()
             if r["city"] == city.lower() and diet.lower() in r["diets"]]
    if not found:
        return f"No {diet} friendly restaurants in {city}."
    return f"{diet} friendly in {city.title()}: " + ", ".join(found)


# Test each tool on its own first
print(find_restaurants("italian", "Riga"))
print(get_restaurant_details("Forno Verde"))
print(make_reservation("Forno Verde", "2026-12-04", 9))
print(filter_by_diet("Riga", "vegan"))

Italian restaurants in Riga: Trattoria Nonna, Forno Verde
Forno Verde: italian, price €, open 11:00-21:00, diets: vegetarian, vegan, tables up to 6 people
Cannot book: Forno Verde takes 1 to 6 people, asked for 9.
vegan friendly in Riga: Forno Verde


In [15]:
restaurant_agent = FunctionAgent(
    tools=[find_restaurants, get_restaurant_details, make_reservation, filter_by_diet],
    llm=llm,
    system_prompt=(
        "You help people choose and book restaurants. Use the tools for every fact. "
        f"Today is {datetime.now():%Y-%m-%d} ({datetime.now():%A})."
    ),
)
# Context(agent) lost the history in the cell above (Alex), so I pass a Memory object instead
from llama_index.core.memory import Memory

restaurant_memory = Memory.from_defaults(session_id="restaurant_chat")


async def ask_restaurant(query: str):
    print(f"🙋 {query}\n")
    handler = restaurant_agent.run(query, memory=restaurant_memory)
    n = 0
    async for event in handler.stream_events():
        if isinstance(event, ToolCallResult):
            n += 1
            print(f"🔧 {n}. {event.tool_name}({event.tool_kwargs}) → {event.tool_output}")
    result = await handler
    print(f"\n💬 {result}\n" + "=" * 60)
    return result


reply = await ask_restaurant("Find Italian restaurants in Riga and get details about the cheaper one.")

🙋 Find Italian restaurants in Riga and get details about the cheaper one.



🔧 1. find_restaurants({'cuisine': 'italian', 'city': 'Riga'}) → Italian restaurants in Riga: Trattoria Nonna, Forno Verde


🔧 2. get_restaurant_details({'restaurant_name': 'Trattoria Nonna'}) → Trattoria Nonna: italian, price €€, open 12:00-23:00, diets: vegetarian, tables up to 8 people


🔧 3. get_restaurant_details({'restaurant_name': 'Forno Verde'}) → Forno Verde: italian, price €, open 11:00-21:00, diets: vegetarian, vegan, tables up to 6 people



💬 I found two Italian restaurants in Riga:

- Trattoria Nonna — €€
- Forno Verde — €

The cheaper one is **Forno Verde**. Here are its details:
- Cuisine: Italian
- Price range: €
- Opening hours: 11:00–21:00
- Diet options: vegetarian, vegan
- Table size: up to 6 people

If you want, I can also help you book a table there.


In [16]:
reply = await ask_restaurant(
    "I'm vegan. Is that place OK for me? If yes, book it for 7 people next Friday."
)

🙋 I'm vegan. Is that place OK for me? If yes, book it for 7 people next Friday.



🔧 1. make_reservation({'restaurant_name': 'Forno Verde', 'date': '2026-11-27', 'party_size': 7}) → Cannot book: Forno Verde takes 1 to 6 people, asked for 7.



💬 Yes — **Forno Verde is OK for vegan diners**.

I tried to book it for **7 people next Friday (2026-11-27)**, but the restaurant can only accommodate **up to 6 people**.

If you want, I can help you:
- book Forno Verde for **6 people**, or
- look for another vegan-friendly Italian restaurant in Riga that can fit **7**.


The second question only says "that place", so it needs the memory from the first one. 7 people is more than Forno Verde takes, so the booking should fail.

In [17]:
reply = await ask_restaurant("OK, then make it 6 people.")
print("Bookings made:", bookings)

🙋 OK, then make it 6 people.



🔧 1. make_reservation({'restaurant_name': 'Forno Verde', 'date': '2026-11-27', 'party_size': 6}) → Booked Forno Verde on 2026-11-27 for 6 people. Confirmation code R001.



💬 Done — **Forno Verde** is booked for **6 people** on **2026-11-27**.

Confirmation code: **R001**
Bookings made: [('R001', 'Forno Verde', '2026-11-27', 6)]


### What I noticed

- `Context(agent)` did not keep the history in this llama-index version. In the Alex demo above the second answer does not know the name or the city, and there is a warning that `memory` was skipped. My agent first had the same problem ("which restaurant is that place?"). With a `Memory` object passed to `run()` the follow-ups work.
- With memory the vegan question needed no new tool call, the agent already knew Forno Verde is vegan from the earlier details. It tried 7 people, my tool refused (max 6), and "make it 6" booked R001 for 2026-11-27 (the coming Friday).
- In the course part, `convert_currency` returns `1000 EUR = 1000.00 JPY` because the pair is missing and the rate falls back to 1.0. The agent passed it on without noticing. A tool should rather return an error.
- For the trips in December the agent asked the weather for today (`days_ahead=0`), not for the flight date, and the answer does not mention which date the weather is for.